<a href="https://colab.research.google.com/github/shreyashan-git/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print('Rows:', len(df))
print('Total revenue:', total_revenue)
print('Total units:', total_units)

Rows: 400
Total revenue: 8520.0
Total units: 783


I basically calculated revenue by multiplying the quantity by the price for each order, then summed the revenue and quantity columns to find that the 400 orders generated $8,478 in total revenue from 788 units sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')['revenue'].sum().reset_index()

by_category['share_of_total'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
).round(1)

by_category = by_category.sort_values(
    'revenue',
    ascending=False
)

print(by_category)

   category  revenue  share_of_total
1      Food   4293.0            50.4
2     Merch   1771.5            20.8
0     Drink   1554.0            18.2
3  RainGear    901.5            10.6


I basically grouped the orders by category and summed the revenue for each one, then calculated each category’s share of total revenue to show which categories contributed the most to overall sales.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
q3 = df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).reset_index()

q3['avg_order_revenue'] = q3['avg_order_revenue'].round(2)

q3 = q3.sort_values('avg_order_revenue', ascending=False)

print(q3)

  vendor_id  avg_order_revenue  order_count
0      V-01              22.60           94
3      V-18              21.75          108
1      V-05              20.58           93
2      V-10              20.31          105


I basically grouped the orders by vendor and calculated the average order revenue along with the number of orders for each vendor so I could identify the vendor with the highest average while also considering how many orders that average represents.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = merch_revenue / df['revenue'].sum() * 100

print(f"{merch_share:.1f}%")

20.8%


I divided the revenue from Merch orders by total revenue to find the percentage of overall revenue that comes from the Merch category.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

rows_before = len(df)
revenue_before = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

rows_after = len(joined)
revenue_after = joined['revenue'].sum()

print('Rows before:', rows_before)
print('Rows after:', rows_after)
print('Revenue before:', revenue_before)
print('Revenue after:', revenue_after)

print('Unmatched vendor:',
      joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique())

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')

Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0
Unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** So basically V-18 was unmatched, so I kept its orders and labeled it as “Unknown Vendor” rather than dropping the sales data.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
q6 = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(q6)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown Vendor    582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


I created a pivot table that summarizes revenue by vendor and category, with row and column totals added to show each vendor’s total revenue and each category’s overall revenue.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

Far a, I would basically tell teh vendors to focus more on Food because it generated 4,293 dollars in revenue, which was much higher than any other category. RainGear only generated 901.50 dollars, so vendors may not need to stock as much of it next game. I would also look at what Hoos Burgers did well with Food since it generated 1,338 dollars from that category, the highest Food revenue among the vendors. For b, basically the vendor-level analysis is the least trustworthy because V-18 did not have a matching vendor name in the lookup table. I had to label V-18 as “Unknown Vendor,” even though it generated $2,349 in revenue, so its hard to say which actual vendor that revenue belongs to.